# 🎵 MusicAI Studio — live test on a free Google Colab GPU

Run every cell top to bottom (**Runtime → Run all**). The last cell prints a public URL — open it on your phone or laptop and generate real AI songs.

**Before running:** go to **Runtime → Change runtime type → T4 GPU** (free tier) or **A100/L4** (Colab Pro, much faster).

In [1]:
# 1) Check which GPU you got
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
# 2) Get the app + install everything (~5-10 minutes)
!git clone https://github.com/ppratik2026/Musicai.git
%cd Musicai
!pip install -q -r requirements.txt
!pip install -q -r requirements-models.txt

Cloning into 'Musicai'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 43 (delta 8), reused 42 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 172.56 KiB | 7.84 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/Musicai
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 23.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.

In [3]:
# 3) Tunnel tool for a public URL (no signup needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
# 4) Start MusicAI + print your public URL
import os, re, subprocess, time

import torch

# T4 (free tier) has no bfloat16 hardware -> use float32 there.
os.environ["MUSICAI_ACE_DTYPE"] = (
    "bfloat16" if torch.cuda.is_bf16_supported() else "float32"
)
print("ACE-Step dtype:", os.environ["MUSICAI_ACE_DTYPE"])

server = subprocess.Popen(["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(8)

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stderr=subprocess.PIPE, text=True,
)
for line in tunnel.stderr:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n" + "=" * 60)
        print("🎵 YOUR LIVE APP:", m.group(0))
        print("=" * 60)
        print("\nOpen the URL, pick the ACE-Step engine, and generate!")
        print("First generation downloads ~7 GB of model weights — be patient.")
        break

## Tips

- **First song is slow** (checkpoint download + model load). After that it's much faster.
- On the free **T4**, start with **30–60 second** tracks. A100/L4 handles full-length songs quickly.
- Keep this tab open — closing Colab stops your server. Download the WAV/MP3 of anything you like before the session ends.
- Songs you plan to **release commercially** must come from the **ACE-Step** engine (Apache-2.0, watermark-free). See `docs/DISTRIBUTION.md` in the repo.